# 01 Preprocess

Prepare `SemEval-2014 Task 4` and `GoEmotions` for the multi-task ABSA + emotion project.

## Workflow

- Load labeled SemEval laptop and restaurant training CSV files
- Keep aspect sentiment classification samples
- Split SemEval into train/dev/test because official PhaseB test files do not include polarity labels
- Load GoEmotions train/dev/test TSV files
- Map original labels to 6 coarse emotions
- Drop ambiguous multi-label or unmapped cases
- Save processed CSV files for later notebooks

In [1]:
from pathlib import Path
import json
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option('display.max_colwidth', 120)

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / 'data'
SEMEVAL_DIR = DATA_DIR / 'semeval'
GOEMOTIONS_DIR = DATA_DIR / 'goemotions' / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_DIR, SEMEVAL_DIR.exists(), GOEMOTIONS_DIR.exists(), PROCESSED_DIR

(WindowsPath('d:/15_MAI/pynotebook/nlpProject'),
 True,
 True,
 WindowsPath('d:/15_MAI/pynotebook/nlpProject/data/processed'))

In [2]:
ABSA_LABEL2ID = {
    'positive': 0,
    'negative': 1,
    'neutral': 2,
    'conflict': 3,
}

EMOTION_LABEL2ID = {
    'joy': 0,
    'anger': 1,
    'sadness': 2,
    'fear': 3,
    'surprise': 4,
    'neutral': 5,
}

ID2ABSA_LABEL = {v: k for k, v in ABSA_LABEL2ID.items()}
ID2EMOTION_LABEL = {v: k for k, v in EMOTION_LABEL2ID.items()}

ABSA_LABEL2ID, EMOTION_LABEL2ID

({'positive': 0, 'negative': 1, 'neutral': 2, 'conflict': 3},
 {'joy': 0, 'anger': 1, 'sadness': 2, 'fear': 3, 'surprise': 4, 'neutral': 5})

## Load SemEval

In [3]:
def load_semeval_csv(path: Path, domain: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df.rename(columns={
        'Sentence': 'text',
        'Aspect Term': 'aspect',
        'polarity': 'label_name',
    })
    df = df[['id', 'text', 'aspect', 'label_name', 'from', 'to']].copy()
    df = df[df['label_name'].isin(ABSA_LABEL2ID)].copy()
    df['label_id'] = df['label_name'].map(ABSA_LABEL2ID)
    df['task'] = 'absa'
    df['domain'] = domain
    return df

semeval_laptop = load_semeval_csv(SEMEVAL_DIR / 'Laptop_Train_v2.csv', 'laptop')
semeval_restaurant = load_semeval_csv(SEMEVAL_DIR / 'Restaurants_Train_v2.csv', 'restaurant')

semeval_df = pd.concat([
    semeval_laptop,
    semeval_restaurant,
], ignore_index=True)

print(semeval_df.shape)
semeval_df.head()

(6051, 9)


,id,text,aspect,label_name,from,to,label_id,task,domain
0,2339,I charge it at night and skip taking the cord with me because of the good battery life.,cord,neutral,41,45,2,absa,laptop
1,2339,I charge it at night and skip taking the cord with me because of the good battery life.,battery life,positive,74,86,0,absa,laptop
2,1316,"The tech guy then said the service center does not do 1-to-1 exchange and I have to direct my concern to the ""sales""...",service center,negative,27,41,1,absa,laptop
3,1316,"The tech guy then said the service center does not do 1-to-1 exchange and I have to direct my concern to the ""sales""...","""sales"" team",negative,109,121,1,absa,laptop
4,1316,"The tech guy then said the service center does not do 1-to-1 exchange and I have to direct my concern to the ""sales""...",tech guy,neutral,4,12,2,absa,laptop


In [4]:
def split_semeval_by_domain(df: pd.DataFrame, random_state: int = 42) -> pd.DataFrame:
    parts = []

    for domain, domain_df in df.groupby('domain'):
        train_df, temp_df = train_test_split(
            domain_df,
            test_size=0.2,
            random_state=random_state,
            stratify=domain_df['label_name']
        )

        dev_df, test_df = train_test_split(
            temp_df,
            test_size=0.5,
            random_state=random_state,
            stratify=temp_df['label_name']
        )

        train_df = train_df.copy()
        dev_df = dev_df.copy()
        test_df = test_df.copy()

        train_df['split'] = 'train'
        dev_df['split'] = 'dev'
        test_df['split'] = 'test'

        parts.extend([train_df, dev_df, test_df])

    return pd.concat(parts, ignore_index=True)

semeval_df = split_semeval_by_domain(semeval_df)
semeval_df.groupby(['domain', 'split', 'label_name']).size().reset_index(name='count')

,domain,split,label_name,count
0,laptop,dev,conflict,4
1,laptop,dev,negative,87
2,laptop,dev,neutral,46
3,laptop,dev,positive,99
4,laptop,test,conflict,5
5,laptop,test,negative,86
6,laptop,test,neutral,46
7,laptop,test,positive,99
8,laptop,train,conflict,36
9,laptop,train,negative,693


In [5]:
semeval_df.groupby(['domain', 'split', 'label_name']).size().reset_index(name='count')

,domain,split,label_name,count
0,laptop,dev,conflict,4
1,laptop,dev,negative,87
2,laptop,dev,neutral,46
3,laptop,dev,positive,99
4,laptop,test,conflict,5
5,laptop,test,negative,86
6,laptop,test,neutral,46
7,laptop,test,positive,99
8,laptop,train,conflict,36
9,laptop,train,negative,693


## Load GoEmotions

In [6]:
emotion_names = [line.strip() for line in (GOEMOTIONS_DIR / 'emotions.txt').read_text(encoding='utf-8').splitlines() if line.strip()]

with open(GOEMOTIONS_DIR / 'ekman_mapping.json', 'r', encoding='utf-8') as f:
    ekman_mapping = json.load(f)

fine_to_coarse = {}
for coarse_label, fine_labels in ekman_mapping.items():
    for fine_label in fine_labels:
        fine_to_coarse[fine_label] = coarse_label

# Merge disgust into anger to keep a 6-class emotion setup.
fine_to_coarse['disgust'] = 'anger'
fine_to_coarse['neutral'] = 'neutral'

emotion_names

['admiration',
 'amusement',
 'anger',
 'annoyance',
 'approval',
 'caring',
 'confusion',
 'curiosity',
 'desire',
 'disappointment',
 'disapproval',
 'disgust',
 'embarrassment',
 'excitement',
 'fear',
 'gratitude',
 'grief',
 'joy',
 'love',
 'nervousness',
 'optimism',
 'pride',
 'realization',
 'relief',
 'remorse',
 'sadness',
 'surprise',
 'neutral']

In [7]:
def parse_goemotions_label_ids(label_str: str):
    parts = [part.strip() for part in str(label_str).split(',') if part.strip() != '']
    return [int(part) for part in parts]

def label_ids_to_names(label_ids):
    return [emotion_names[idx] for idx in label_ids]

def map_to_single_coarse_label(label_ids):
    fine_names = label_ids_to_names(label_ids)
    coarse_names = {fine_to_coarse[name] for name in fine_names if name in fine_to_coarse}

    if len(coarse_names) == 1:
        coarse_name = next(iter(coarse_names))
        return coarse_name

    return None

def load_goemotions_split(path: Path, split: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep='\t', header=None, names=['text', 'raw_labels', 'example_id'])
    df['label_ids'] = df['raw_labels'].apply(parse_goemotions_label_ids)
    df['fine_labels'] = df['label_ids'].apply(label_ids_to_names)
    df['label_name'] = df['label_ids'].apply(map_to_single_coarse_label)
    df = df[df['label_name'].notna()].copy()
    df['label_id'] = df['label_name'].map(EMOTION_LABEL2ID)
    df = df[df['label_id'].notna()].copy()
    df['label_id'] = df['label_id'].astype(int)
    df['task'] = 'emotion'
    df['split'] = split
    return df

go_train = load_goemotions_split(GOEMOTIONS_DIR / 'train.tsv', 'train')
go_dev = load_goemotions_split(GOEMOTIONS_DIR / 'dev.tsv', 'dev')
go_test = load_goemotions_split(GOEMOTIONS_DIR / 'test.tsv', 'test')

goemotions_df = pd.concat([go_train, go_dev, go_test], ignore_index=True)

print(goemotions_df.shape)
goemotions_df.head()

(49469, 9)


,text,raw_labels,example_id,label_ids,fine_labels,label_name,label_id,task,split
0,My favourite food is anything I didn't have to cook myself.,27,eebbqej,[27],[neutral],neutral,5.0,emotion,train
1,"Now if he does off himself, everyone will think hes having a laugh screwing with people instead of actually dead",27,ed00q6i,[27],[neutral],neutral,5.0,emotion,train
2,WHY THE FUCK IS BAYLESS ISOING,2,eezlygj,[2],[anger],anger,1.0,emotion,train
3,To make her feel threatened,14,ed7ypvh,[14],[fear],fear,3.0,emotion,train
4,Dirty Southern Wankers,3,ed0bdzj,[3],[annoyance],anger,1.0,emotion,train


In [8]:
goemotions_df.groupby(['split', 'label_name']).size().reset_index(name='count')

,split,label_name,count
0,dev,anger,555
1,dev,disgust,61
2,dev,fear,72
3,dev,joy,1941
4,dev,neutral,1592
5,dev,sadness,266
6,dev,surprise,459
7,test,anger,572
8,test,disgust,76
9,test,fear,80


In [9]:
raw_go_train = pd.read_csv(GOEMOTIONS_DIR / 'train.tsv', sep='\t', header=None, names=['text', 'raw_labels', 'example_id'])
raw_go_train['label_ids'] = raw_go_train['raw_labels'].apply(parse_goemotions_label_ids)
raw_go_train['coarse_label'] = raw_go_train['label_ids'].apply(map_to_single_coarse_label)

kept_count = raw_go_train['coarse_label'].notna().sum()
dropped_count = raw_go_train['coarse_label'].isna().sum()

print('GoEmotions train kept:', kept_count)
print('GoEmotions train dropped:', dropped_count)

GoEmotions train kept: 39555
GoEmotions train dropped: 3855


## Build unified task format

In [10]:
absa_final = semeval_df[['task', 'domain', 'split', 'id', 'text', 'aspect', 'label_name', 'label_id']].copy()
absa_final = absa_final.rename(columns={'id': 'sample_id'})

emotion_final = goemotions_df[['task', 'split', 'example_id', 'text', 'label_name', 'label_id']].copy()
emotion_final['domain'] = 'reddit'
emotion_final['aspect'] = ''
emotion_final = emotion_final.rename(columns={'example_id': 'sample_id'})
emotion_final = emotion_final[['task', 'domain', 'split', 'sample_id', 'text', 'aspect', 'label_name', 'label_id']]

absa_final.head(), emotion_final.head()

(   task  domain  split  sample_id  \
 0  absa  laptop  train       1964   
 1  absa  laptop  train        462   
 2  absa  laptop  train        908   
 3  absa  laptop  train       1895   
 4  absa  laptop  train        963   
 
                                                                                                                       text  \
 0                                                                    This computer was so challenging to carry and handle.   
 1                                                                                 Unable to boot up this brand new laptop.   
 2  Also, my sister got the exact same laptop (since they were so cheap) and after 8 months, the screen split in half ju...   
 3  Overall I feel this netbook was poor quality, had poor performance, although it did have great battery life when it ...   
 4                                    Another THREE weeks later I had my laptop back with a new mousepad, keys, and casing.   
 
      

In [11]:
print('ABSA shape:', absa_final.shape)
print('Emotion shape:', emotion_final.shape)

display(absa_final.sample(5, random_state=42))
display(emotion_final.sample(5, random_state=42))

ABSA shape: (6051, 8)
Emotion shape: (49469, 8)


,task,domain,split,sample_id,text,aspect,label_name,label_id
4876,absa,restaurant,train,838,The aesthetics of this place are like an airport lounge.,place,negative,1
3114,absa,restaurant,train,3355,"- for dessert we split chocolate cake and vanilla gelato (with espresso), which were tasty, but I thought a bit over...",vanilla gelato (with espresso),conflict,3
3518,absa,restaurant,train,1652,"Otherwise, this place has great service and prices and a nice friendly atmosphere.",atmosphere,positive,0
2299,absa,laptop,test,1449,"The apple systems are over priced luxurys that arn't worth what they are being charged for, this model's specificati...",specifications,negative,1
1694,absa,laptop,train,1740,The Material this Pro is made out of seems a lot nicer than any PC Specs: Like I said this performs a lot better th...,Specs,negative,1


,task,domain,split,sample_id,text,aspect,label_name,label_id
10075,emotion,reddit,train,ed3axr6,From one eating disorder to another. That's not recovery that's just sad.,,sadness,2.0
3502,emotion,reddit,train,eeedg85,"Thats nice to hear, glad I could help.",,joy,0.0
9613,emotion,reddit,train,ee4or8t,It’s more like a large youtuber trying to sue a smaller youtuber because both of their names have a C in it.,,neutral,5.0
25787,emotion,reddit,train,ee15at7,Big thanks to OP for that magnificent wall. I guess that we can reopen the government now.,,joy,0.0
31077,emotion,reddit,train,ef8ilhi,Everyone is sad on this sub dont take it personally if you don't get any replies.,,sadness,2.0


## Save processed files

In [12]:
absa_final.to_csv(PROCESSED_DIR / 'absa_semeval_2014.csv', index=False)
emotion_final.to_csv(PROCESSED_DIR / 'emotion_goemotions_6class.csv', index=False)

metadata = {
    'absa_label2id': ABSA_LABEL2ID,
    'emotion_label2id': EMOTION_LABEL2ID,
    'goemotions_fine_to_coarse': fine_to_coarse,
}

with open(PROCESSED_DIR / 'label_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

list(PROCESSED_DIR.iterdir())

[WindowsPath('d:/15_MAI/pynotebook/nlpProject/data/processed/absa_semeval_2014.csv'),
 WindowsPath('d:/15_MAI/pynotebook/nlpProject/data/processed/emotion_goemotions_6class.csv'),
 WindowsPath('d:/15_MAI/pynotebook/nlpProject/data/processed/label_metadata.json')]